## Hometask

Objective: Apply at least two (2) modern sentiment analysis methods to classify text data and evaluate their performance

Dataset:

Sentiment Analysis Dataset: https://www.cs.cornell.edu/people/pabo/movie-review-data/rt-polaritydata.tar.gz

alternative source: rt-polaritydata

Each line in these two files corresponds to a single snippet (usually containing roughly one single sentence); all snippets are down-cased.

More info about dataset

rt-polarity.neg: Contains negative reviews.
rt-polarity.pos: Contains positive reviews
Task Description:
Data Loading & Preparation:

Load rt-polarity.neg and rt-polarity.pos.
Split each file into individual snippets.
Assign labels (0 for negative, 1 for positive).
Combine into a single dataset.
Split the dataset into training and testing sets.
Implement and evaluate at least three (3) methods from the lecture, prioritizing modern approaches.

For each implemented method, report and compare classification metrics: Accuracy, Precision, Recall, and F1-score.

In [ ]:
import pandas as pd
import numpy as np
import random
import nltk


from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import gensim
from gensim.models import Word2Vec

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import torch

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\LEGION 5
[nltk_data]     PRO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
fn='data/rt-polarity.neg'

with open(fn, "r",encoding='utf-8', errors='ignore') as f: # some invalid symbols encountered
    content = f.read()
texts_neg=  content.splitlines()
print ('len of texts_neg = {:,}'.format (len(texts_neg)))
for review in texts_neg[:5]:
    print ( '\n', review)

len of texts_neg = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [ ]:
fn='data/rt-polarity.pos'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_pos=  content.splitlines()
print ('len of texts_pos = {:,}'.format (len(texts_pos)))
for review in texts_pos[:5]:
    print ('\n', review)

len of texts_pos = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


## countvectorizer + logistic regression

In [6]:
labels_neg = [0] * len(texts_neg)
labels_pos = [1] * len(texts_pos)

X = texts_neg + texts_pos
y = labels_neg + labels_pos

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nРозмір тренувального набору: {len(X_train)}")
print(f"Розмір тестового набору: {len(X_test)}")


Розмір тренувального набору: 8529
Розмір тестового набору: 2133


In [7]:
vectorizer = CountVectorizer(min_df=5, ngram_range=(1,2))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"Розмірність векторизованих тренувальних даних: {X_train_vectorized.shape}")
print(f"Розмірність векторизованих тестових даних: {X_test_vectorized.shape}")

Розмірність векторизованих тренувальних даних: (8529, 7065)
Розмірність векторизованих тестових даних: (2133, 7065)


In [9]:
model_count_lr = LogisticRegression(max_iter=5000, random_state=42)

model_count_lr.fit(X_train_vectorized, y_train)

LogisticRegression(max_iter=5000, random_state=42)

In [10]:
y_pred_count_lr = model_count_lr.predict(X_test_vectorized)

In [12]:
accuracy = accuracy_score(y_test, y_pred_count_lr)
precision = precision_score(y_test, y_pred_count_lr)
recall = recall_score(y_test, y_pred_count_lr)
f1 = f1_score(y_test, y_pred_count_lr)

print("\nРезультати для CountVectorizer + Logistic Regression:\n")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")


Результати для CountVectorizer + Logistic Regression:

Accuracy:  0.7623
Precision: 0.7600
Recall:    0.7664
F1-score:  0.7632


## Word2Vec

In [17]:
def tokenize_text(text):
    return word_tokenize(text)


X_train_tokenized = [tokenize_text(text) for text in X_train]
X_test_tokenized = [tokenize_text(text) for text in X_test]

In [42]:
w2v_model = Word2Vec(sentences=X_train_tokenized,
                     vector_size=50,
                     window=5,
                     min_count=2,
                     workers=-1)

print(f"Vocabulary size: {len(w2v_model.wv.key_to_index)}")

Vocabulary size: 8823


In [43]:
if "good" in w2v_model.wv:
    print("\nСлова, найбільш схожі на 'good':")
    for word, score in w2v_model.wv.most_similar("good", topn=5):
        print(f"  {word}: {score:.4f}")


Слова, найбільш схожі на 'good':
  started: 0.4658
  pete: 0.4597
  willis: 0.4567
  elsewhere: 0.4556
  griffin: 0.4421


Крутив параметри, слова мягко кажучи не дуже схожі) спишемо це на величину датасету

In [44]:
def document_vector(doc_tokens, model):
    words_in_vocab = [token for token in doc_tokens if token in model.wv]

    if not words_in_vocab:
        return np.zeros(model.vector_size)
        
    return np.mean([model.wv[word] for word in words_in_vocab], axis=0)

In [45]:
X_train_w2v_vectors = np.array([document_vector(doc, w2v_model) for doc in X_train_tokenized])
X_test_w2v_vectors = np.array([document_vector(doc, w2v_model) for doc in X_test_tokenized])

In [51]:
w2v_clf = LogisticRegression(max_iter=10000)
w2v_clf.fit(X_train_w2v_vectors, y_train)

w2v_predictions = w2v_clf.predict(X_test_w2v_vectors)
w2v_scores = w2v_clf.predict_proba(X_test_w2v_vectors)[:, 1]

accuracy = accuracy_score(y_test, w2v_predictions)
precision = precision_score(y_test, w2v_predictions)
recall = recall_score(y_test, w2v_predictions)
f1 = f1_score(y_test, w2v_predictions)

print("\nWord2Vec + Logistic Regression")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


--- Word2Vec + Logistic Regression ---
Accuracy: 0.5546
Precision: 0.5506
Recall: 0.5919
F1 Score: 0.5705


ЖАХ

## Transformer-Based Models (distilbert-base-uncased-finetuned-sst-2-english)

In [60]:
from transformers import pipeline
import torch.nn.functional as F

In [73]:
sample_size_train_transformer = 5000
sample_size_test_transformer = 1000

indices_train_tf = np.random.choice(len(X_train), min(len(X_train), sample_size_train_transformer), replace=False)
X_train_sample_tf = [X_train[i] for i in indices_train_tf]
y_train_sample_tf = [y_train[i] for i in indices_train_tf]

indices_test_tf = np.random.choice(len(X_test), min(len(X_test), sample_size_test_transformer), replace=False)
X_test_sample_tf = [X_test[i] for i in indices_test_tf]
y_test_sample_tf = [y_test[i] for i in indices_test_tf]

train_dataset_tf = Dataset.from_dict({
    'text': X_train_sample_tf,
    'label': y_train_sample_tf
})

test_dataset_tf = Dataset.from_dict({
    'text': X_test_sample_tf,
    'label': y_test_sample_tf
})

In [76]:
model_name_distilbert = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer_distilbert = AutoTokenizer.from_pretrained(model_name_distilbert)
model_distilbert = AutoModelForSequenceClassification.from_pretrained(model_name_distilbert, num_labels=2)

def tokenize_function_transformer(examples):
    return tokenizer_distilbert(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train_tf = train_dataset_tf.map(tokenize_function_transformer, batched=True)
tokenized_test_tf = test_dataset_tf.map(tokenize_function_transformer, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [77]:
tokenized_train_tf = tokenized_train_tf.remove_columns(["text"])
tokenized_test_tf = tokenized_test_tf.remove_columns(["text"])

tokenized_train_tf = tokenized_train_tf.rename_column("label", "labels")
tokenized_test_tf = tokenized_test_tf.rename_column("label", "labels")

tokenized_train_tf.set_format("torch")
tokenized_test_tf.set_format("torch")

In [79]:
def compute_metrics_transformer(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    precision = precision_score(labels, predictions, average='binary')
    recall = recall_score(labels, predictions, average='binary')
    f1 = f1_score(labels, predictions, average='binary')
    accuracy = accuracy_score(labels, predictions)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [81]:
training_args_tf = TrainingArguments(
    output_dir="./results_distilbert_sst2",       
    num_train_epochs=3,                         
    per_device_train_batch_size=16,             
    per_device_eval_batch_size=64,              
    warmup_steps=100,                           
    weight_decay=0.01,                          
    logging_dir="./logs_distilbert_sst2",       
    logging_steps=50,                           
    evaluation_strategy="epoch",                
    save_strategy="epoch",                
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"                 
    # no_cuda=True
)

trainer_distilbert = Trainer(
    model=model_distilbert,
    args=training_args_tf,
    train_dataset=tokenized_train_tf,
    eval_dataset=tokenized_test_tf, 
    tokenizer=tokenizer_distilbert, 
    compute_metrics=compute_metrics_transformer 
)


trainer_distilbert.train()

C:\Users\LEGION 5 PRO\AppData\Local\Temp\ipykernel_3680\4029139929.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_distilbert = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.285200,0.321811,0.889000,0.887080,0.906445,0.868526
2,0.162000,0.400168,0.890000,0.891089,0.885827,0.896414
3,0.047700,0.597729,0.892000,0.893701,0.883268,0.904382


TrainOutput(global_step=939, training_loss=0.16742380574361457, metrics={'train_runtime': 2691.6916, 'train_samples_per_second': 5.573, 'train_steps_per_second': 0.349, 'total_flos': 496752744960000.0, 'train_loss': 0.16742380574361457, 'epoch': 3.0})

як і очікувалось моделька з huggingface дала топ результати